# Qiskit basics: circuits and local simulation

This notebook is adapted from the now-deprecated [`qiskit-tutorials`](https://github.com/Qiskit/qiskit-tutorials) repository and has been updated for modern Qiskit versions.

The original material is distributed under the following license:

> © Copyright IBM 2017, 2021.  
> Licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).


## Local simulation with Qiskit Aer

We will use the [`qiskit-aer`](https://qiskit.github.io/qiskit-aer/) package for local statevector and shot-based simulations.


### Packages

This notebook uses the following packages:

- `qiskit` (versions 2.1.2 and 2.3.0 have been tested and work fine)
- `qiskit-aer` (version 0.17.2)
- `pylatexenc` (needed only for Matplotlib circuit drawings)

If the packages are already installed in your Python environment, you can skip the next cell. In Google Colab or a fresh environment, run it once before continuing. Restart the kernel if the environment asks you to do so.


In [ ]:
%pip install --quiet "qiskit==2.1.2" "qiskit-aer==0.17.2" pylatexenc

## Circuit basics <a name="basics"></a>

### Building the circuit

The basic object used to construct a quantum program in Qiskit is `QuantumCircuit`. We begin by creating a circuit containing three qubits.


In [ ]:
from qiskit import QuantumCircuit, transpile

# Create a quantum circuit acting on three qubits.
circ = QuantumCircuit(3)

After creating the circuit, we can add gates (operations) that act on its qubits. The following example prepares the three-qubit GHZ state

$$
|\psi\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}.
$$

By default, every qubit is initialized in the state $|0\rangle$. To prepare the GHZ state, we apply:

- a Hadamard gate $H$ to qubit $q_0$, producing the superposition $(|0\rangle+|1\rangle)/\sqrt{2}$;
- a controlled-X gate (CX, also called CNOT) with $q_0$ as control and $q_1$ as target;
- a controlled-X gate with $q_0$ as control and $q_2$ as target.

On an ideal quantum computer, the resulting state is the GHZ state written above.


In [ ]:
# Put qubit q_0 into a superposition.
circ.h(0)

# Entangle q_1 and q_2 with q_0.
circ.cx(0, 1)
circ.cx(0, 2)

### Visualizing the circuit <a name="visualize"></a>

The method `QuantumCircuit.draw()` displays the circuit using the notation commonly found in quantum-computing textbooks.


In [ ]:
# Matplotlib output; this requires pylatexenc.
circ.draw(output="mpl")

A text-based diagram is also available and does not require additional visualization packages.


In [ ]:
print(circ.draw(output="text"))

In the default circuit diagram, qubit $q_0$ is at the top and qubit $q_2$ is at the bottom. The circuit is read from left to right: operations shown further to the left are applied earlier.


<div class="alert alert-block alert-info">
<b>Bit-ordering convention.</b>

Qiskit labels the qubits as $q_0,q_1,\ldots,q_{n-1}$. In circuit diagrams, $q_0$ is drawn at the top. In displayed bitstrings and computational-basis kets, however, the highest-index qubit is written on the left and $q_0$ on the right:

$$
|q_{n-1}\cdots q_1q_0\rangle.
$$

For example, if $q_0=0$, $q_1=0$, and $q_2=1$, Qiskit represents the state as $|100\rangle$.

This convention also determines the matrix representation of multi-qubit gates. For example, Qiskit represents a controlled-X gate with $q_0$ as control and $q_1$ as target as

$$
C_X =
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0 \\
0 & 1 & 0 & 0
\end{pmatrix}.
$$
</div>


## Simulating circuits with Qiskit Aer <a name="simulation"></a>

Qiskit Aer provides high-performance local simulators. In this notebook, we use the `AerSimulator` class.

### Statevector simulation

A statevector simulation returns the quantum state as a complex vector of dimension $2^n$, where $n$ is the number of qubits. The required memory therefore grows exponentially with the number of qubits.


We first create an `AerSimulator` configured to use the statevector method.


In [ ]:
from qiskit_aer import AerSimulator

# Configure a local statevector simulator.
sim_statevector = AerSimulator(method="statevector")

Aer returns a final statevector only if the circuit contains an instruction that saves it. To keep the original circuit unchanged, we make a copy and append `save_statevector()` to the copy.

The save instruction must be placed before any measurement if we want the uncollapsed pre-measurement state.


In [ ]:
statevector_circ = circ.copy()
statevector_circ.save_statevector()

statevector_circ.draw(output="mpl")

Before execution, we transpile the circuit for the selected backend. The backend's `run()` method then returns a job object representing the submitted simulation.

In [ ]:
compiled_statevector_circ = transpile(statevector_circ, sim_statevector)
job = sim_statevector.run(compiled_statevector_circ)

A job object provides, among others, the methods `job.status()` and `job.result()`. The first reports the job status; the second waits for completion when necessary and returns a result object.


In [ ]:
job.status()

In [ ]:
result = job.result()

The result object contains the data produced by the simulation. The method `result.get_statevector(...)` retrieves the saved statevector for the circuit.


In [ ]:
output_state = result.get_statevector(compiled_statevector_circ, decimals=3)
print(output_state)

Qiskit also provides visualization tools for quantum states. The following function displays the real and imaginary parts of the density matrix associated with the statevector.


In [ ]:
from qiskit.visualization import plot_state_city

plot_state_city(output_state)

### Measurements and shot-based simulation


Statevector simulation gives direct access to the ideal quantum state. A physical experiment, however, ends by measuring the qubits, usually in the computational basis $\{|0\rangle,|1\rangle\}$. Each measurement produces classical bits and, in the usual projective-measurement description, collapses the quantum state.

Consider independent measurements of the three qubits in the GHZ state

$$
|\psi\rangle = \frac{|000\rangle+|111\rangle}{\sqrt{2}}.
$$

Let $xyz$ denote the displayed measurement bitstring. With Qiskit's bit-ordering convention, $x$ is the outcome of qubit $q_2$, $y$ the outcome of $q_1$, and $z$ the outcome of $q_0$.

<div class="alert alert-block alert-info">
<b>Note:</b> Qubit $q_0$ is the least significant bit and appears on the right of the displayed bitstring; the highest-index qubit appears on the left.
</div>

The probability of obtaining the outcome $xyz$ is

$$
\Pr(xyz)=|\langle xyz|\psi\rangle|^2.
$$

For the GHZ state, the only possible ideal outcomes are $000$ and $111$, each with probability $1/2$.

To simulate measurements, we add classical bits and measurement operations to the circuit.


In [ ]:
# Create a circuit with three qubits and three classical bits.
mcirc = QuantumCircuit(3, 3)

# Prepare the same GHZ state.
mcirc.h(0)
mcirc.cx(0, 1)
mcirc.cx(0, 2)

# The barrier is only a visual separator here.
mcirc.barrier()

# Store the measurement of qubit q_j in classical bit c_j.
mcirc.measure(range(3), range(3))

mcirc.draw(output="mpl")

The circuit now contains a classical register and three measurements. Each shot produces one classical bitstring. Repeating the circuit many times builds an empirical distribution of the outcomes.

The number of repetitions is specified by the `shots` argument of `run()`.


In [ ]:
# Use a general Aer simulator for shot-based execution.
sim_shots = AerSimulator()
compiled_mcirc = transpile(mcirc, sim_shots)

job_shots = sim_shots.run(compiled_mcirc, shots=1024)
result_shots = job_shots.result()

The method `get_counts(...)` returns the aggregated measurement outcomes as a dictionary: each key is a measured bitstring, and its value is the number of times that bitstring occurred.


In [ ]:
counts = result_shots.get_counts(compiled_mcirc)
print(counts)

The two observed counts should be close to one another, although they will generally not be exactly equal because the number of shots is finite. Qiskit provides `plot_histogram()` to visualize the distribution.


In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts)

<div class="alert alert-block alert-warning">
<b>Exercise.</b> Change the value of `shots` in the call to `run()` and observe how the estimated probabilities change. For each outcome, estimate the probability by dividing its count by the total number of shots.
</div>
